# Chapter 23: Exploring and Summarizing Data**Companion notebook** for *Beginner's Guide to Pandas* by Ravi ShankarRun each cell in order. Exercises are at the end.

In [ ]:
import pandas as pdimport numpy as np

# Exploring and Summarizing DataOnce you've loaded your data, it's time to explore and summarize it. Pandas provides powerful tools for understanding distributions, relationships, and outliers in your dataset. This chapter covers practical techniques to systematically examine your data and extract meaningful initial insights — the critical foundation before any modeling or visualization work.---## Inspecting Your DataBefore diving into statistics, you need to understand the structure and quality of your data. Start by answering fundamental questions: What does this data contain? How large is it? What types of variables are present?

In [ ]:
import pandas as pd
import numpy as np

# Generate comprehensive college.csv with all columns used in this chapter
np.random.seed(42)
n = 30
regions = ['Midwest', 'South', 'Northeast', 'West']
controls = ['Public', 'Private', 'For-Profit']

college_df = pd.DataFrame({
    'INSTNM': [f'University of {chr(65+i)}{chr(97+j)}' for i in range(n) for j in range(1)][:n],
    'TUITION': np.random.randint(5000, 75000, n).astype(float),
    'ENROLLMENT': np.random.randint(100, 45000, n).astype(float),
    'GRAD_RATE': np.random.uniform(20, 98, n).round(1),
    'REGION': np.random.choice(regions, n),
    'CONTROL': np.random.choice(controls, n),
    'date_column': pd.date_range('2020-01-01', periods=n, freq='ME').astype(str),
    'region_column': np.random.choice(regions, n),
})

# Inject some missing values to make the data realistic for EDA exercises
for col in ['TUITION', 'ENROLLMENT', 'GRAD_RATE']:
    mask = np.random.choice(n, size=int(n * 0.1), replace=False)
    college_df.loc[mask, col] = np.nan

# Add a duplicate row for duplicate detection exercises
college_df = pd.concat([college_df, college_df.iloc[[0]]], ignore_index=True)

college_df.to_csv('college.csv', index=False)

# Load the data
college = pd.read_csv('college.csv')  # Replace with your file path

# View the first few rows
print(college.head())

# View the last few rows
print(college.tail(3))

# Get information about data types and missing values
print(college.info())

# Shows: column names, data types, non-null counts, and memory usage

# Check for missing values
print(college.isnull().sum())

# Example output:
#   INSTNM          0
#   TUITION         5
#   ENROLLMENT      2
#   GRAD_RATE      12

# Check the shape of your data
print(college.shape)  # Returns (rows, columns)

The `info()` method is particularly valuable because it immediately reveals potential data quality issues like missing values or unexpected data types. The `isnull().sum()` call tells you exactly how many values are missing in each column.**Why this matters:** Missing values (NaN) can silently affect calculations. Pandas automatically excludes NaN from most operations (like `.mean()`, `.sum()`), but you need to know they exist before you can decide how to handle them.**Alternative data sources:**

In [ ]:
# Also create Excel and JSON versions from the same data
college.to_excel('college.xlsx', index=False)
college.to_json('college.json', orient='records', indent=2)

college = pd.read_excel('college.xlsx')  # Excel files
college = pd.read_json('college.json')   # JSON files

### Understanding Data TypesOnce you know the shape of your data, inspect the types of each column:

In [ ]:
# Check data types of each columnprint(college.dtypes)# Get a summary of how many columns have each typeprint(college.dtypes.value_counts())# Convert data types if neededcollege['date_column'] = pd.to_datetime(college['date_column'])college['region_column'] = college['region_column'].astype('category')

Correct data types matter: a numeric column stored as strings won't support arithmetic, and a categorical column stored as plain strings uses more memory than necessary.### Checking for Duplicate Records

In [ ]:
# Check for complete duplicatesprint(f"Complete duplicates: {college.duplicated().sum()}")# Check for duplicates based on specific columnsprint(f"Duplicates on key columns: {college.duplicated(subset=['INSTNM']).sum()}")# Examine duplicate rowsduplicates = college[college.duplicated(subset=['INSTNM'], keep=False)].sort_values('INSTNM')print(duplicates.head(10))# Remove duplicatescollege_clean = college.drop_duplicates(subset=['INSTNM'], keep='first')

---## Descriptive Statistics### Understanding `.describe()`The `.describe()` method gives you a quick statistical portrait of every numeric column at once.

In [ ]:
import pandas as pdcollege = pd.read_csv('college.csv')# Summary statistics for all numeric columnsprint(college.describe())# Include specific percentilesprint(college.describe(percentiles=[.25, .5, .75, .90]))# Statistics for specific columnsprint(college[['TUITION', 'GRAD_RATE']].describe())# Include non-numeric columnsprint(college.describe(include='all'))

**Output example:**```         TUITION   ENROLLMENT   GRAD_RATEcount    1234.00      1234.00     1222.00mean    28500.50     4521.30       65.40std     15200.30     3210.50       12.80min      5000.00      100.00       20.0025%     18000.00     2100.00       58.0050%     27500.00     4200.00       65.0075%     38000.00     6500.00       72.00max     75000.00    45000.00       98.00```**What each statistic means:**| Statistic | Meaning | Why It Matters ||-----------|---------|----------------|| **count** | Non-null values | Identifies missing data || **mean** | Average value | Overall typical value || **std** | Standard deviation | Spread of the data (higher = more variation) || **min** | Smallest value | Lower bound || **25%** (Q1) | First quartile | 25% below, 75% above || **50%** (median) | Middle value | Robust average (resistant to outliers) || **75%** (Q3) | Third quartile | 75% below, 25% above || **max** | Largest value | Upper bound |**Key insight:** Compare **mean** vs. **median** to detect skewness. If mean > median, data is right-skewed (a few very high values pull the average up). If mean < median, data is left-skewed.### Additional Distribution MeasuresBeyond `.describe()`, you can compute skewness and kurtosis to characterize the shape of a distribution:

In [ ]:
# Skewness: positive = right-skewed, negative = left-skewedprint(f"Skewness: {college['TUITION'].skew()}")# Kurtosis: how heavy the tails are relative to a normal distributionprint(f"Kurtosis: {college['TUITION'].kurtosis()}")# Coefficient of variation: std as a fraction of the meancv = college['TUITION'].std() / college['TUITION'].mean()print(f"Coefficient of Variation: {cv:.2f}")

---### Counting Values with `.value_counts()`

In [ ]:
import pandas as pdcollege = pd.read_csv('college.csv')# Count occurrences of each categoryprint(college['REGION'].value_counts())# Output:#   Midwest      450#   South        380#   Northeast    220#   West         184# Show as percentagesprint(college['REGION'].value_counts(normalize=True))# Output:#   Midwest      0.365  (36.5%)#   South        0.308  (30.8%)#   Northeast    0.178  (17.8%)#   West         0.149  (14.9%)# Sort in ascending orderprint(college['REGION'].value_counts(ascending=True))# Get top 3 categoriesprint(college['REGION'].value_counts().head(3))# Include NaN values in the countprint(college['REGION'].value_counts(dropna=False))

You can also build a combined count-and-percentage summary, which is useful for spotting rare categories:

In [ ]:
value_dist = college['REGION'].value_counts()value_dist_pct = college['REGION'].value_counts(normalize=True) * 100summary = pd.DataFrame({    'Count': value_dist,    'Percentage': value_dist_pct})print(summary)# Flag rare categories (less than 1% of data)print("\nRare categories (< 1%):")print(summary[summary['Percentage'] < 1])

**When to use:** `.value_counts()` is perfect for **categorical columns** to understand how observations are distributed across categories.---### Computing Mean, Median, and Mode

In [ ]:
import pandas as pdcollege = pd.read_csv('college.csv')# Mean (average)print(college['TUITION'].mean())# Median (middle value, robust to outliers)print(college['TUITION'].median())# Mode (most common value) — returns a Series in case of tiesprint(college['TUITION'].mode())# Extract the single mode value safelymode_values = college['TUITION'].mode()if len(mode_values) > 0:    print(f"Mode: {mode_values[0]}")else:    print("No mode found")

**Intuition:**- **Mean** works well for symmetric data but can be pulled by outliers- **Median** is more robust when you have extreme values- **Mode** finds the most common value (especially useful for categorical data)---## Handling Missing DataMissing values are a reality in most datasets. Understanding their patterns is crucial before deciding how to handle them.

In [ ]:
import pandas as pdcollege = pd.read_csv('college.csv')# Count of missing values per columnprint(college.isnull().sum())# Percentage of missing values per columnmissing_pct = (college.isnull().sum() / len(college)) * 100print(missing_pct[missing_pct > 0])# Find rows with any missing valuesrows_with_missing = college[college.isnull().any(axis=1)]print(f"Rows with at least one missing value: {len(rows_with_missing)}")

### Strategies for Missing Data

In [ ]:
# Drop rows with missing values in any column
college_clean = college.dropna()

# Drop rows only where a specific column is missing
college_clean = college.dropna(subset=['GRAD_RATE'])

# Fill missing values with a constant
college['TUITION'] = college['TUITION'].fillna(0)

# Fill with column mean (common for numeric columns)
college['TUITION'] = college['TUITION'].fillna(college['TUITION'].mean())

# Forward fill (propagate last valid value — useful for time series)
college['TUITION'] = college['TUITION'].ffill()

**Choosing a strategy:** Dropping rows is safe when missing data is rare. Filling with the mean preserves sample size but can reduce variance. Always document which strategy you chose and why.---## Grouping and Aggregation### The "Split-Apply-Combine" ConceptGroupby follows three steps:1. **Split:** Divide data into groups based on a column2. **Apply:** Calculate a statistic for each group3. **Combine:** Merge results back together

In [ ]:
import pandas as pdcollege = pd.read_csv('college.csv')# Average graduation rate by regionprint(college.groupby('REGION')['GRAD_RATE'].mean())# Output:#   REGION#   Midwest      86.0#   Northeast    97.5#   South        72.3#   West         81.5# Maximum enrollment by regionprint(college.groupby('REGION')['ENROLLMENT'].max())# Count of colleges per regionprint(college.groupby('REGION').size())

### Multiple Aggregations at Once

In [ ]:
import pandas as pdcollege = pd.read_csv('college.csv')# Apply multiple functions to one columnprint(college.groupby('REGION')['TUITION'].agg(['mean', 'min', 'max', 'std']))# Apply different functions to different columnsprint(college.groupby('REGION').agg({    'TUITION': ['mean', 'min', 'max'],    'ENROLLMENT': 'sum',    'GRAD_RATE': 'median'}))# Use custom column names for clarityprint(college.groupby('REGION')['TUITION'].agg(    avg_tuition='mean',    min_tuition='min',    max_tuition='max'))

You can also define a custom aggregation function when the built-in options aren't enough:

In [ ]:
def coefficient_of_variation(series):    return pd.Series({        'mean': series.mean(),        'std': series.std(),        'cv': series.std() / series.mean()  # Relative variability    })college.groupby('REGION')['TUITION'].apply(coefficient_of_variation)

### Combining Filtering and Grouping

In [ ]:
import pandas as pdcollege = pd.read_csv('college.csv')# Average graduation rate by region for affordable colleges (tuition < $30,000)print(college[college['TUITION'] < 30000].groupby('REGION')['GRAD_RATE'].mean())# Count colleges per region with enrollment > 5000print(college[college['ENROLLMENT'] > 5000].groupby('REGION').size())# Use .copy() if you'll modify the filtered dataaffordable_colleges = college[college['TUITION'] < 30000].copy()affordable_colleges['PRICE_CATEGORY'] = 'Affordable'

---## Exploring Relationships Between Variables### Correlation AnalysisCorrelation measures how strongly two numeric variables move together. Values range from -1 (perfect negative relationship) to +1 (perfect positive relationship).

In [ ]:
import pandas as pdcollege = pd.read_csv('college.csv')# Correlation matrix for all numeric columnscorr_matrix = college.corr(numeric_only=True)print(corr_matrix)# Correlation of all columns with one target variableprint(college.corr(numeric_only=True)['GRAD_RATE'].sort_values(ascending=False))

To find pairs of variables that are highly correlated with each other (which can cause problems in modeling):

In [ ]:
import numpy as npdef find_high_correlations(corr_matrix, threshold=0.8):    high_corr = []    for i in range(len(corr_matrix.columns)):        for j in range(i + 1, len(corr_matrix.columns)):            if abs(corr_matrix.iloc[i, j]) > threshold:                high_corr.append({                    'var1': corr_matrix.columns[i],                    'var2': corr_matrix.columns[j],                    'correlation': corr_matrix.iloc[i, j]                })    return pd.DataFrame(high_corr)high_correlations = find_high_correlations(corr_matrix)print(high_correlations)

### Cross-Tabulation for Categorical Variables

In [ ]:
# Create a contingency tablecrosstab = pd.crosstab(college['REGION'], college['CONTROL'])# With row and column totalscrosstab = pd.crosstab(college['REGION'], college['CONTROL'], margins=True)# Normalize to see row proportionscrosstab_pct = pd.crosstab(college['REGION'], college['CONTROL'], normalize='index') * 100print(crosstab_pct.round(1))

---## Detecting Outliers and Trends### Method 1: Fixed Threshold (Simple but Arbitrary)

In [ ]:
import pandas as pdcollege = pd.read_csv('college.csv')# Find colleges with unusually high tuitionoutliers = college[college['TUITION'] > 60000]print(outliers[['INSTNM', 'TUITION']])

**Limitation:** The threshold (60000) is arbitrary and doesn't adapt to your data's actual spread.---### Method 2: IQR Method (More Robust)The **Interquartile Range (IQR)** method is statistically sound and adapts to your data:

In [ ]:
import pandas as pdcollege = pd.read_csv('college.csv')# Calculate quartilesQ1 = college['TUITION'].quantile(0.25)Q3 = college['TUITION'].quantile(0.75)IQR = Q3 - Q1print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")# Define outlier boundaries (standard: 1.5 × IQR)lower_bound = Q1 - 1.5 * IQRupper_bound = Q3 + 1.5 * IQRprint(f"Outlier range: {lower_bound} to {upper_bound}")# Find outliersoutliers = college[    ((college['TUITION'] < lower_bound) |     (college['TUITION'] > upper_bound)) &    (college['TUITION'].notna())]print(f"Found {len(outliers)} outliers out of {len(college)} colleges")print(outliers[['INSTNM', 'TUITION']])

You can wrap this logic in a reusable function to apply it across multiple columns:

In [ ]:
def identify_outliers(series):    Q1 = series.quantile(0.25)    Q3 = series.quantile(0.75)    IQR = Q3 - Q1    lower_bound = Q1 - 1.5 * IQR    upper_bound = Q3 + 1.5 * IQR    return (series < lower_bound) | (series > upper_bound)# Apply to multiple columnsfor col in ['TUITION', 'ENROLLMENT', 'GRAD_RATE']:    outlier_mask = identify_outliers(college[col])    print(f"\n{col}: {outlier_mask.sum()} outliers detected")    print(college[outlier_mask][col].describe())

**Why IQR is better:** It's based on the actual spread of your data. The 1.5 × IQR threshold captures approximately 99.3% of data in a normal distribution.---### Method 3: Z-Score Method (For Normally Distributed Data)

In [ ]:
import pandas as pdimport numpy as npfrom scipy import statscollege = pd.read_csv('college.csv')# Calculate z-scores (standard deviations from the mean)college['TUITION_ZSCORE'] = np.abs(stats.zscore(college['TUITION'].fillna(college['TUITION'].mean())))# Outliers typically have z-scores > 3outliers = college[college['TUITION_ZSCORE'] > 3]print(outliers[['INSTNM', 'TUITION', 'TUITION_ZSCORE']])# More sensitive threshold: z-score > 2 (captures ~95% of normal data)outliers_sensitive = college[college['TUITION_ZSCORE'] > 2]

**When to use:** Z-score works best when your data is approximately normally distributed. Use IQR if you're unsure about the distribution.---## Visualizing PatternsPictures reveal patterns that numbers hide. Here are key visualizations for EDA:

In [ ]:
import pandas as pdimport matplotlib.pyplot as pltcollege = pd.read_csv('college.csv')# Distribution of tuition (histogram)college['TUITION'].hist(bins=30, edgecolor='black')plt.title('Distribution of Tuition')plt.xlabel('Tuition ($)')plt.ylabel('Number of Colleges')plt.show()# Box plot to see median, quartiles, and outlierscollege.boxplot(column='TUITION', by='REGION')plt.title('Tuition by Region')plt.suptitle('')  # Remove automatic titleplt.show()# Value counts as bar chartcollege['REGION'].value_counts().plot(kind='bar')plt.title('Number of Colleges by Region')plt.xlabel('Region')plt.ylabel('Count')plt.show()# Grouped statisticscollege.groupby('REGION')['GRAD_RATE'].mean().plot(kind='bar')plt.title('Average Graduation Rate by Region')plt.xlabel('Region')plt.ylabel('Graduation Rate (%)')plt.show()# Scatter plot to see relationshipsplt.scatter(college['TUITION'], college['GRAD_RATE'], alpha=0.5)plt.xlabel('Tuition ($)')plt.ylabel('Graduation Rate (%)')plt.title('Tuition vs. Graduation Rate')plt.show()# Missing data patternscollege.isnull().sum().plot(kind='barh')plt.title('Missing Values per Column')plt.xlabel('Count')plt.show()

If you have the `seaborn` library available, you can visualize the correlation matrix as a heatmap:

In [ ]:
import seaborn as snscorr_matrix = college.corr(numeric_only=True)plt.figure(figsize=(10, 8))sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0)plt.title('Correlation Matrix')plt.tight_layout()plt.show()

**Choose your visualization by question:**| Question | Best Visualization ||----------|-------------------|| How is a numeric column distributed? | Histogram or box plot || Which categories are most common? | Bar chart || How do groups compare? | Grouped bar chart or box plot || Is there a relationship between two columns? | Scatter plot || How correlated are numeric variables? | Heatmap || Where is data missing? | Bar chart of null counts || Are there outliers? | Box plot or scatter plot |---## Common Pitfalls and Tips

In [ ]:
import pandas as pdimport numpy as npcollege = pd.read_csv('college.csv')# ✅ RIGHT: Use .copy() when filtering to avoid SettingWithCopyWarningfiltered = college[college['TUITION'] < 30000].copy()filtered['NEW_COLUMN'] = 100# ❌ WRONG: Modifying filtered data without .copy()# filtered = college[college['TUITION'] < 30000]# filtered['NEW_COLUMN'] = 100  # May raise a warning or silently fail# ✅ RIGHT: Check for missing values before computing statisticsprint(f"Missing values: {college['TUITION'].isnull().sum()}")print(f"Mean TUITION: {college['TUITION'].mean()}")  # Automatically excludes NaN# ✅ RIGHT: Use vectorized groupby operations (fast)result = college.groupby('REGION')['TUITION'].mean()# ❌ WRONG: Use loops (slow)# result_slow = {}# for region in college['REGION'].unique():#     result_slow[region] = college[college['REGION'] == region]['TUITION'].mean()# ✅ RIGHT: Check if mode exists before accessingmode_values = college['TUITION'].mode()if len(mode_values) > 0:    print(f"Mode: {mode_values[0]}")else:    print("No mode found")

**Key takeaways:**- Use `.copy()` after filtering to avoid warnings when modifying data- `.mean()` and similar functions automatically handle NaN values- Vectorized operations (`.groupby()`) are much faster than loops- Always check data types and missing values before any analysis---## Generating a Summary ReportOnce you've explored your data, it's useful to consolidate findings into a structured summary you can share or revisit:

In [ ]:
import pandas as pdimport numpy as npcollege = pd.read_csv('college.csv')def identify_outliers(series):    Q1 = series.quantile(0.25)    Q3 = series.quantile(0.75)    IQR = Q3 - Q1    return (series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)def generate_eda_report(df):    """Generate a comprehensive EDA summary."""    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()    report = {        'Dataset Shape': df.shape,        'Total Missing Values': df.isnull().sum().sum(),        'Duplicate Rows': df.duplicated().sum(),        'Numerical Columns': numeric_cols,        'Categorical Columns': categorical_cols,        'Memory Usage (MB)': round(df.memory_usage(deep=True).sum() / 1024**2, 2)    }    for col in numeric_cols:        report[f'{col}_skewness'] = round(df[col].skew(), 3)        report[f'{col}_outlier_count'] = int(identify_outliers(df[col]).sum())    return reporteda_report = generate_eda_report(college)for key, value in eda_report.items():    print(f"{key}: {value}")

---## SummaryExploring and summarizing your data helps you spot patterns, errors, and opportunities for deeper analysis. Follow this workflow:1. **Inspect** with `.info()`, `.head()`, `.isnull().sum()`, and `.duplicated()`2. **Describe** with `.describe()`, `.value_counts()`, and skewness/kurtosis3. **Handle missing data** by deciding whether to drop or fill, and documenting your choice4. **Group** with `.groupby()` to find trends across categories5. **Explore relationships** with correlation matrices and cross-tabulations6. **Detect outliers** using IQR or z-scores7. **Visualize** patterns with histograms, box plots, scatter plots, and heatmapsThis foundation prepares you for more advanced analysis and modeling. EDA is not a one-time activity — return to it as your understanding of the data deepens, and document your findings as you go.

---# ExercisesTest your understanding of this chapter's concepts.

### Exercise 1: Inspect and Describe a Sales DatasetCreate a small sales DataFrame and use basic inspection methods to understand its structure. Practice using head(), info(), shape, and describe() to get an overview of the data.

In [ ]:
import pandas as pd# Sample sales datadata = {    'product': ['Widget', 'Gadget', 'Widget', 'Doohickey', 'Gadget', 'Widget', 'Doohickey', 'Gadget'],    'region': ['North', 'South', 'East', 'North', 'East', 'South', 'East', 'North'],    'units_sold': [120, 85, 200, 45, 310, 95, 60, 175],    'revenue': [2400.0, 4250.0, 4000.0, 900.0, 15500.0, 1900.0, 1200.0, 8750.0]}df = pd.DataFrame(data)# TODO: Print the first 5 rows of the DataFrame# TODO: Print the shape of the DataFrame (rows, columns)# TODO: Print the data types and non-null counts using info()# TODO: Print descriptive statistics for all numeric columns using describe()

### Exercise 2: Handle Missing Data in an Employee DatasetWork with an employee DataFrame that contains missing values. Identify where missing data exists, calculate the percentage of missing values per column, and fill or drop them appropriately.

In [ ]:
import pandas as pd# Employee data with missing valuesdata = {    'name': ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank'],    'department': ['HR', 'Engineering', None, 'Engineering', 'HR', None],    'salary': [55000, 92000, 78000, None, 61000, 85000],    'years_exp': [3, 7, None, 5, 2, None]}df = pd.DataFrame(data)# TODO: Print the count of missing values in each column using isnull().sum()# TODO: Print the percentage of missing values per column#       (hint: divide isnull().sum() by len(df) and multiply by 100)# TODO: Fill missing values in 'department' with the string 'Unknown'# TODO: Fill missing values in 'salary' and 'years_exp' with the median of each column# TODO: Print the cleaned DataFrame to confirm no missing values remain

### Exercise 3: Group and Aggregate Store Transaction DataUse groupby and aggregation functions to summarize a store transactions dataset. Calculate total revenue, average units sold, and transaction count broken down by product category and region.

In [ ]:
import pandas as pd# Store transaction datadata = {    'category': ['Electronics', 'Clothing', 'Electronics', 'Food', 'Clothing', 'Food', 'Electronics', 'Clothing', 'Food'],    'region': ['East', 'West', 'West', 'East', 'East', 'West', 'East', 'West', 'East'],    'units_sold': [5, 12, 3, 30, 8, 25, 7, 15, 40],    'revenue': [1500, 360, 900, 150, 240, 125, 2100, 450, 200]}df = pd.DataFrame(data)# TODO: Group by 'category' and calculate the total revenue and mean units_sold#       Assign the result to a variable called category_summary# TODO: Group by both 'category' and 'region' and calculate:#       - total revenue (sum)#       - average units_sold (mean)#       - number of transactions (count) for the 'units_sold' column#       Use .agg() with a dictionary. Assign result to region_summary# TODO: Print both summary DataFrames

### Exercise 4: Detect Outliers and Summarize RelationshipsAnalyze a health metrics dataset to detect outliers using the IQR method and explore the correlation between numeric variables. Identify which records are outliers and print a correlation matrix to reveal relationships.

In [ ]:
import pandas as pd# Health metrics datasetdata = {    'age': [25, 32, 47, 51, 23, 38, 60, 29, 45, 55],    'bmi': [22.1, 27.5, 31.2, 24.8, 19.5, 28.9, 35.4, 21.0, 30.1, 200.0],    'blood_pressure': [120, 135, 145, 130, 118, 140, 160, 122, 138, 128],    'cholesterol': [180, 210, 240, 195, 170, 225, 260, 185, 230, 215]}df = pd.DataFrame(data)# TODO: Calculate Q1, Q3, and IQR for the 'bmi' column# TODO: Define lower and upper bounds as Q1 - 1.5*IQR and Q3 + 1.5*IQR# TODO: Filter and print the rows where 'bmi' is outside the bounds (outliers)# TODO: Print the correlation matrix for all numeric columns using .corr()#       Round the result to 2 decimal places

---# Solutions*Scroll down only after you've attempted the exercises above.*<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Inspect and Describe a Sales Dataset

In [ ]:
import pandas as pdimport numpy as np# Sample sales datadata = {    'product': ['Widget', 'Gadget', 'Widget', 'Doohickey', 'Gadget', 'Widget', 'Doohickey', 'Gadget'],    'region': ['North', 'South', 'East', 'North', 'East', 'South', 'East', 'North'],    'units_sold': [120, 85, 200, 45, 310, 95, 60, 175],    'revenue': [2400.0, 4250.0, 4000.0, 900.0, 15500.0, 1900.0, 1200.0, 8750.0]}df = pd.DataFrame(data)# Print the first 5 rows of the DataFrameprint("First 5 rows:")print(df.head())# Print the shape of the DataFrame (rows, columns)print("\nShape:", df.shape)# Print the data types and non-null counts using info()print("\nDataFrame Info:")df.info()# Print descriptive statistics for all numeric columns using describe()print("\nDescriptive Statistics:")print(df.describe())

### Solution 2: Handle Missing Data in an Employee Dataset

In [ ]:
import pandas as pdimport numpy as np# Employee data with missing valuesdata = {    'name': ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank'],    'department': ['HR', 'Engineering', None, 'Engineering', 'HR', None],    'salary': [55000, 92000, 78000, None, 61000, 85000],    'years_exp': [3, 7, None, 5, 2, None]}df = pd.DataFrame(data)# Print the count of missing values in each columnprint("Missing value counts:")print(df.isnull().sum())# Print the percentage of missing values per columnprint("\nMissing value percentages:")print((df.isnull().sum() / len(df)) * 100)# Fill missing values in 'department' with 'Unknown'df['department'] = df['department'].fillna('Unknown')# Fill missing values in 'salary' and 'years_exp' with the median of each columndf['salary'] = df['salary'].fillna(df['salary'].median())df['years_exp'] = df['years_exp'].fillna(df['years_exp'].median())# Print the cleaned DataFrameprint("\nCleaned DataFrame:")print(df)print("\nRemaining missing values:")print(df.isnull().sum())

### Solution 3: Group and Aggregate Store Transaction Data

In [ ]:
import pandas as pdimport numpy as np# Store transaction datadata = {    'category': ['Electronics', 'Clothing', 'Electronics', 'Food', 'Clothing', 'Food', 'Electronics', 'Clothing', 'Food'],    'region': ['East', 'West', 'West', 'east', 'East', 'West', 'East', 'West', 'East'],    'units_sold': [5, 12, 3, 30, 8, 25, 7, 15, 40],    'revenue': [1500, 360, 900, 150, 240, 125, 2100, 450, 200]}df = pd.DataFrame(data)# Group by 'category' and calculate total revenue and mean units_soldcategory_summary = df.groupby('category').agg(    total_revenue=('revenue', 'sum'),    avg_units_sold=('units_sold', 'mean'))# Group by 'category' and 'region' with multiple aggregationsregion_summary = df.groupby(['category', 'region']).agg(    total_revenue=('revenue', 'sum'),    avg_units_sold=('units_sold', 'mean'),    transaction_count=('units_sold', 'count'))# Print both summary DataFramesprint("Category Summary:")print(category_summary)print("\nCategory + Region Summary:")print(region_summary)

### Solution 4: Detect Outliers and Summarize Relationships

In [ ]:
import pandas as pdimport numpy as np# Health metrics datasetdata = {    'age': [25, 32, 47, 51, 23, 38, 60, 29, 45, 55],    'bmi': [22.1, 27.5, 31.2, 24.8, 19.5, 28.9, 35.4, 21.0, 30.1, 200.0],    'blood_pressure': [120, 135, 145, 130, 118, 140, 160, 122, 138, 128],    'cholesterol': [180, 210, 240, 195, 170, 225, 260, 185, 230, 215]}df = pd.DataFrame(data)# Calculate Q1, Q3, and IQR for the 'bmi' columnQ1 = df['bmi'].quantile(0.25)Q3 = df['bmi'].quantile(0.75)IQR = Q3 - Q1print(f"BMI Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")# Define lower and upper boundslower_bound = Q1 - 1.5 * IQRupper_bound = Q3 + 1.5 * IQRprint(f"Lower bound: {lower_bound:.2f}, Upper bound: {upper_bound:.2f}")# Filter and print rows where 'bmi' is outside the boundsbmi_outliers = df[(df['bmi'] < lower_bound) | (df['bmi'] > upper_bound)]print("\nBMI Outliers:")print(bmi_outliers)# Print the correlation matrix rounded to 2 decimal placesprint("\nCorrelation Matrix:")print(df.corr().round(2))